| Existing calendar integrations | Temporal Cortex |
|---|---|
| Basic CRUD (create/read/update/delete events) | 18 tools across 5 layers with deterministic computation |
| Single provider (Google Calendar only) | Cross-provider merging (Google + Outlook + CalDAV) |
| No conflict detection | Two-Phase Commit atomic booking |
| LLM guesses timezone/datetime | `resolve_datetime` + `get_temporal_context` — zero hallucination |
| No recurrence rule support | Truth Engine: RFC 5545 expansion (DST, BYSETPOS, EXDATE) |
| No availability merging | `get_availability` merges free/busy across all calendars |

## What Temporal Cortex Adds

| Existing calendar integrations | Temporal Cortex |
|---|---|
| Basic CRUD (create/read/update/delete events) | 18 tools across 5 layers with deterministic computation |
| Single provider (Google Calendar only) | Cross-provider merging (Google + Outlook + CalDAV) |
| No conflict detection | Two-Phase Commit atomic booking |
| LLM guesses timezone/datetime | `resolve_datetime` + `get_temporal_context` — zero hallucination |
| No recurrence rule support | Truth Engine: RFC 5545 expansion (DST, BYSETPOS, EXDATE) |
| No availability merging | `get_availability` merges free/busy across all calendars |

Temporal Cortex is an [MCP server](https://github.com/temporal-cortex/mcp) — CrewAI's native `MCPServerAdapter` auto-discovers all tools with zero custom code.

In [ ]:
!pip install crewai crewai-tools[mcp] python-dotenv

## Setup

Before running this notebook:

1. **Authenticate with your calendar provider** (run in terminal):
   ```bash
   npx @temporal-cortex/cortex-mcp auth google
   ```
   This opens a browser for OAuth and stores credentials locally.

2. **Set your environment variables** below (or create a `.env` file):
   - `GOOGLE_CLIENT_ID` — from [Google Cloud Console](https://console.cloud.google.com/apis/credentials)
   - `GOOGLE_CLIENT_SECRET` — from the same page

   See the [Google Calendar setup guide](https://github.com/temporal-cortex/mcp/blob/main/docs/google-cloud-setup.md) for detailed instructions. Microsoft Outlook and CalDAV are also supported.

In [ ]:
import os

from crewai import Agent, Crew, Process, Task
from crewai_tools import MCPServerAdapter
from dotenv import load_dotenv

load_dotenv()

# MCP server configuration — stdio transport (runs locally via npx)
server_params = {
    "command": "npx",
    "args": ["-y", "@temporal-cortex/cortex-mcp"],
    "env": {
        "GOOGLE_CLIENT_ID": os.getenv("GOOGLE_CLIENT_ID", ""),
        "GOOGLE_CLIENT_SECRET": os.getenv("GOOGLE_CLIENT_SECRET", ""),
        "TIMEZONE": os.getenv("TIMEZONE", ""),
    },
}

## Step 1: Connect and Discover Tools

The `MCPServerAdapter` connects to the Temporal Cortex MCP server and auto-discovers all available tools. No manual tool registration needed.

Temporal Cortex organizes tools in 5 layers:

| Layer | Purpose | Tools |
|-------|---------|-------|
| **1 — Temporal Context** | Orient in time | `get_temporal_context`, `resolve_datetime`, `convert_timezone`, `compute_duration`, `adjust_timestamp` |
| **2 — Calendar Ops** | Query calendars | `list_calendars`, `list_events`, `find_free_slots`, `expand_rrule`, `check_availability` |
| **3 — Availability** | Merged free/busy | `get_availability` |
| **4 — Booking** | Atomic booking | `book_slot` |

In [ ]:
# Connect to the MCP server and list discovered tools
adapter = MCPServerAdapter(server_params)
tools = adapter.start()

print(f"Discovered {len(tools)} Temporal Cortex tools:\n")
for tool in tools:
    print(f"  - {tool.name}: {tool.description[:80]}...")

## Step 2: Define Agents

We create 3 specialized agents that mirror the tool layer architecture. Each agent's **backstory encodes the correct tool-calling order** — this is how we guide agents to use the right tools without hard-coding tool selection.

| Agent | Layer | Role |
|-------|-------|------|
| **Temporal Analyst** | Layer 1 | Orients in time — calls `get_temporal_context` then `resolve_datetime` |
| **Calendar Manager** | Layers 2-3 | Queries calendars — calls `list_calendars` then `find_free_slots` |
| **Scheduling Coordinator** | Layer 4 | Books meetings — calls `book_slot` with Two-Phase Commit safety |

In [ ]:
temporal_analyst = Agent(
    role="Temporal Analyst",
    goal=(
        "Establish temporal context by determining the current time, "
        "timezone, and DST status, then resolve any human datetime "
        "expressions into precise RFC 3339 timestamps"
    ),
    backstory=(
        "You are a time-awareness specialist. Before any calendar "
        "operation, you call get_temporal_context to learn the current "
        "time, timezone, UTC offset, and DST status. You convert "
        "natural language like 'next Tuesday at 2pm' into exact "
        "timestamps using resolve_datetime. You never guess dates "
        "or times — you always use the tools."
    ),
    tools=tools,
    verbose=True,
)

calendar_manager = Agent(
    role="Calendar Manager",
    goal=(
        "Query connected calendars to list events, find free time "
        "slots, and check availability across all providers"
    ),
    backstory=(
        "You are a calendar operations specialist. You list calendars "
        "to discover connected providers (Google, Outlook, CalDAV), "
        "query events in time ranges, and find available slots using "
        "find_free_slots. You always use provider-prefixed calendar "
        "IDs (e.g., google/primary, outlook/work) and pass RFC 3339 "
        "timestamps from the Temporal Analyst. You never assume "
        "calendar IDs — you discover them first with list_calendars."
    ),
    tools=tools,
    verbose=True,
)

coordinator = Agent(
    role="Scheduling Coordinator",
    goal=(
        "Coordinate the scheduling workflow: use temporal context and "
        "availability data to book a conflict-free meeting"
    ),
    backstory=(
        "You are the lead scheduler. You take the resolved timestamps "
        "from the Temporal Analyst and the available slots from the "
        "Calendar Manager to select the best meeting time. You use "
        "book_slot to create the event — this tool uses Two-Phase "
        "Commit to acquire a lock, verify no conflicts exist, write "
        "the event, then release the lock. You never double-book."
    ),
    tools=tools,
    verbose=True,
)

print("Agents created: Temporal Analyst, Calendar Manager, Scheduling Coordinator")

## Step 3: Define Tasks

Tasks follow the **orient → query → book** workflow:

1. **Orient** — Get current time, resolve "next Tuesday at 2pm" to an exact timestamp
2. **Find Availability** — Discover calendars, find free 30-minute slots on the target date
3. **Book Meeting** — Select the best slot and book with atomic Two-Phase Commit

Each task passes context to the next via the `context` parameter — the Calendar Manager receives the resolved timestamp from the Temporal Analyst, and the Coordinator receives both.

In [ ]:
# Task 1: Orient in time (Layer 1)
orient_task = Task(
    description=(
        "Get the current temporal context: call get_temporal_context "
        "to learn the current time, timezone, UTC offset, and DST "
        "status. Then resolve the meeting time expression "
        "'next Tuesday at 2pm' into a precise RFC 3339 timestamp "
        "using resolve_datetime."
    ),
    expected_output=(
        "The current local time, timezone, DST status, and the "
        "resolved RFC 3339 timestamp for 'next Tuesday at 2pm'."
    ),
    agent=temporal_analyst,
)

# Task 2: Find available slots (Layers 2-3)
availability_task = Task(
    description=(
        "Using the resolved timestamp from the previous task, "
        "find available time slots on the target date. First call "
        "list_calendars to discover connected providers and their "
        "calendar IDs. Then call find_free_slots for the primary "
        "calendar on the target date to find 30-minute available "
        "windows. Report all available slots."
    ),
    expected_output=(
        "A list of connected calendars and available 30-minute time "
        "slots on the target date, with start/end times in RFC 3339 "
        "format and the calendar ID to use for booking."
    ),
    agent=calendar_manager,
    context=[orient_task],
)

# Task 3: Book the meeting (Layer 4)
booking_task = Task(
    description=(
        "From the available slots found in the previous task, "
        "select the slot closest to 2pm and book a 30-minute meeting "
        "titled 'Team Sync'. Use the book_slot tool with the "
        "calendar ID from the Calendar Manager. The book_slot tool "
        "uses Two-Phase Commit to prevent double-bookings."
    ),
    expected_output=(
        "Confirmation that the meeting was booked, including the "
        "calendar ID, event title, start time, and end time."
    ),
    agent=coordinator,
    context=[orient_task, availability_task],
)

print("Tasks created: Orient -> Find Availability -> Book Meeting")

## Step 4: Run the Crew

In [ ]:
crew = Crew(
    agents=[temporal_analyst, calendar_manager, coordinator],
    tasks=[orient_task, availability_task, booking_task],
    process=Process.sequential,
    verbose=True,
)

result = crew.kickoff()
print("\n" + "=" * 60)
print("SCHEDULING RESULT")
print("=" * 60)
print(result)

- **No Node.js or npx required** — connects directly via SSE
- **Managed OAuth lifecycle** — no local credential files to manage
- **Distributed locking** — safe for multiple agents booking simultaneously
- **18 tools** (3 additional Open Scheduling tools for agent-to-agent booking)
- **Content firewall** — prompt injection detection and sanitization
- **Usage dashboard** — monitor tool calls and calendar connections

## Upgrade: Platform Mode (SSE)

The example above runs locally via `npx`. For production deployments, connect to the **Temporal Cortex Platform** instead:

- **No Node.js or npx required** — connects directly via SSE
- **Managed OAuth lifecycle** — no local credential files to manage
- **Distributed locking** — safe for multiple agents booking simultaneously
- **18 tools** (3 additional Open Scheduling tools for agent-to-agent booking)
- **Content firewall** — prompt injection detection and sanitization
- **Usage dashboard** — monitor tool calls and calendar connections

Sign up at [app.temporal-cortex.com](https://app.temporal-cortex.com) and get your API key.

In [ ]:
# Platform Mode — SSE transport (replace the server_params above)

# from crewai_tools import MCPServerAdapter
#
# platform_params = {
#     "url": "https://mcp.temporal-cortex.com/mcp",
#     "transport": "sse",
#     "headers": {
#         "Authorization": f"Bearer {os.getenv('TEMPORAL_CORTEX_API_KEY', '')}",
#     },
# }
#
# with MCPServerAdapter(platform_params) as tools:
#     print(f"Discovered {len(tools)} tools (including Open Scheduling)")
#     # ... same agent/task/crew setup as above

# --- OR use the DSL approach (simplest possible integration) ---

# from crewai import Agent
# from crewai.mcp import MCPServerStdio
#
# scheduler = Agent(
#     role="Calendar Scheduling Assistant",
#     goal="Schedule meetings using deterministic calendar tools",
#     backstory="You always call get_temporal_context first to orient in time.",
#     mcps=[
#         MCPServerStdio(
#             command="npx",
#             args=["-y", "@temporal-cortex/cortex-mcp"],
#             env={"TIMEZONE": "America/New_York"},
#         ),
#     ],
# )

print("See examples above — uncomment the variant you want to use.")

In [ ]:
- **[Tool Reference](https://github.com/temporal-cortex/mcp/blob/main/docs/tools.md)** — Complete input/output schemas for all 18 tools

## Next Steps

- **[Temporal Cortex MCP](https://github.com/temporal-cortex/mcp)** — Full documentation, setup guides, and architecture overview
- **[Tool Reference](https://github.com/temporal-cortex/mcp/blob/main/docs/tools.md)** — Complete input/output schemas for all 18 tools
- **[Agent Skills](https://github.com/temporal-cortex/skills)** — Install procedural knowledge for calendar workflows (`npx skills add temporal-cortex/skills`)
- **[Platform Dashboard](https://app.temporal-cortex.com)** — Sign up for managed hosting, OAuth lifecycle, and usage analytics
- **[CrewAI Integration Guide](https://github.com/temporal-cortex/mcp/blob/main/docs/crewai-integration.md)** — Detailed guide with transport options and troubleshooting